# method

> The ways you learn: code, maths, analogies, worked examples, diagrams and plain words. Which work for you, and when to try a new one.

Each question, hint and lesson uses one way to teach, its `method`. The tutor picks the method before it writes the prompt, so the tag is
known, not guessed. When a question closes, its outcome counts for its method: solid 1, shaky 0.5, a miss 0.

The counts give each method a Beta posterior, for each subject and over all subjects. `choose` uses them in two ways:

- **Drill.** A weak concept gets your best method.
- **Explore.** A concept that is not weak sometimes gets another method: first a method that got better lately, else a random draw
  (a Thompson sample), with the chance that you set (`explore` in the profile).

The counts live in one `stats` document on the shelf. It is only a cache: `method_stats` builds it again from the attempts.

In [ ]:
#| default_exp method

In [ ]:
#| hide
from fastcore.test import *
from abhyasa.store import temp_learning

In [ ]:
#| export
import json, random, threading, time
from fastcore.utils import *
from abhyasa.store import Learning, _src

## The methods

`METHODS` maps each method to the line that the prompt gets. `EXAMPLES_TO` maps the profile's "examples that help" to methods, for the prior.

In [ ]:
#| export
METHODS = dict(code='Use a short Python code example that shows the idea; keep it under 12 lines.',
               maths='Use equations and formal notation; derive the step, one line at a time.',
               analogy="Use an analogy from the learner's own world (see 'About the learner'), and say where it stops working.",
               example='Use one concrete worked example, step by step, with real numbers or real words.',
               diagram='Describe a picture in words: what to draw, where each part goes, and what moves.',
               words='Use plain words only: no code and no equations.')
LABELS = dict(code='Code', maths='Maths', analogy='Analogies', example='Worked examples', diagram='Pictures', words='Plain words')
EXAMPLES_TO = {'code': 'code', 'maths': 'maths', 'analogies': 'analogy', 'stories': 'analogy', 'diagrams': 'diagram', 'real cases': 'example'}
CREDIT = dict(solid=1., fragile=.5, gap=0., misconception=0.)
EXPLORE = dict(never=0., sometimes=.15, often=.3)
_SRC, _LOCK = _src('stats', 'me'), threading.Lock()

## The stats document

`stats` reads it, with empty counts when there is none. `set_stats` writes it. The lock keeps two closes from losing a count.

In [ ]:
#| export
EMPTY = dict(methods={}, recent={}, calib={})

@patch
def stats(self:Learning) -> AttrDict:
    "The cached stats: method counts for each subject (`*` is all), recent credits, and calibration."
    r = first(self.v.t.docs(where='source=?', where_args=[_SRC]))
    return AttrDict({**EMPTY, **(json.loads(r['meta'] or '{}') if r else {})})

@patch
def set_stats(self:Learning, **kv):
    "Change fields of the stats document."
    m = {**{k: v for k, v in self.stats().items() if k in EMPTY}, **kv}
    r = self.v.add('How you learn: method counts.', title='How you learn', source=_SRC, kind='stats', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)

## Credit

`credit` adds one outcome to the counts and returns the new stats. It keeps the last 8 credits of each method, to see a method that gets better.

In [ ]:
#| export
def credit(st, method:str, subject:str, kind:str) -> dict:
    "The stats `st` with one outcome `kind` added for `method`, in `subject` and in `*`."
    s, ms = CREDIT.get(kind), {k: {m: list(ab) for m, ab in v.items()} for k, v in (st.get('methods') or {}).items()}
    if s is None or method not in METHODS: return dict(st)
    for k in {subject or '*', '*'}:
        a, b = ms.setdefault(k, {}).get(method, [0., 0.])
        ms[k][method] = [a + s, b + 1 - s]
    rc = {**(st.get('recent') or {})}
    rc[method] = [*rc.get(method, []), s][-8:]
    return {**st, 'methods': ms, 'recent': rc}

@patch
def credit_method(self:Learning, method:str, subject:str, kind:str):
    "Count outcome `kind` for `method` in the stats document."
    with _LOCK: self.set_stats(**{k: v for k, v in credit(self.stats(), method, subject, kind).items() if k in EMPTY})

Solid adds 1 to `a`; a miss adds 1 to `b`; shaky adds half to each. An unknown outcome or method changes nothing.

In [ ]:
st = credit(credit(credit(EMPTY, 'code', 'ml', 'solid'), 'code', 'ml', 'fragile'), 'maths', 'ml', 'gap')
test_eq(st['methods']['ml'], {'code': [1.5, .5], 'maths': [0., 1.]})
test_eq(st['methods']['*'], st['methods']['ml'])
test_eq(st['recent'], {'code': [1., .5], 'maths': [0.]})
test_eq(credit(st, 'code', 'ml', 'closed'), st)

## The posterior

`posterior` gives `(a, b)` for a method: Beta(1, 1), plus 2 on `a` for a method in the profile's examples. For a subject it adds the subject counts and the `*` counts, scaled down to at most 4 pseudo-counts, so a new subject starts from what works for you elsewhere.

In [ ]:
#| export
def posterior(st, m:str, subject:str='', prior=()) -> tuple:
    "The Beta `(a, b)` of method `m` in `subject`."
    ms = st.get('methods') or {}
    a, b = 1. + 2 * (m in prior), 1.
    g = (ms.get('*') or {}).get(m, [0., 0.])
    if subject and (s := (ms.get(subject) or {}).get(m)):
        k = min(1., 4 / max(sum(g), 1e-9))
        return a + s[0] + k * g[0], b + s[1] + k * g[1]
    return a + g[0], b + g[1]

def mean(ab) -> float: return ab[0] / sum(ab)

def improving(st, m:str) -> tuple|None:
    "`(right, n)` over the last 5 credits of `m`, when they are at least 0.2 above its mean over all; else None."
    r, ab = (st.get('recent') or {}).get(m, []), (st.get('methods') or {}).get('*', {}).get(m)
    if len(r) < 4 or not ab or sum(ab) < 4: return None
    last = r[-5:]
    return (round(sum(last)), len(last)) if sum(last) / len(last) >= ab[0] / sum(ab) + .2 else None

With no counts, the prior decides. A method that got better lately is found.

In [ ]:
test_eq(posterior(EMPTY, 'code'), (1., 1.))
test_eq(posterior(EMPTY, 'code', prior=['code']), (3., 1.))
gm = dict(methods={'*': {'maths': [2., 4.]}}, recent={'maths': [0., 0., 1., 1., 1., 1.]})
test_eq(improving(gm, 'maths'), (4, 5))
test_is(improving(dict(gm, recent={'maths': [1., 0., 0., 0.]}), 'maths'), None)

## Choose a method

`choose` returns `(method, note)`. The note goes in the line that says why you get this question.

In [ ]:
#| export
def _count(st, m, subject=''):
    "The raw `(right, n)` of method `m`, in `subject` when it has counts there, else over all."
    ms = st.get('methods') or {}
    a, b = (ms.get(subject) or {}).get(m) or (ms.get('*') or {}).get(m) or [0., 0.]
    return round(a), round(a + b)

def choose(st, subject:str='', weak:bool=False, rng=None, explore:str='sometimes', prior=()) -> tuple:
    "The method for the next question, and a note on why. A weak concept gets the best method; else, sometimes, another one."
    rng = rng or random
    ps = {m: posterior(st, m, subject, prior) for m in METHODS}
    best = max(ps, key=lambda m: (mean(ps[m]), m in prior))
    if weak:
        k, n = _count(st, best, subject)
        return best, f"drill with {LABELS[best].lower()}: {k} of {n}, your best way" if n else (f"drill with {LABELS[best].lower()}: a way that you said helps" if best in prior else '')
    if (up := first((m, improving(st, m)) for m in METHODS if m != best and improving(st, m))):
        m, (k, n) = up
        if rng.random() < max(EXPLORE.get(explore, .15) * 2, .3) * bool(EXPLORE.get(explore, .15)):
            return m, f"trying {LABELS[m].lower()}: you got better with it ({k} of {n} lately)"
    if rng.random() < EXPLORE.get(explore, .15):
        m = max((m for m in METHODS if m != best), key=lambda m: rng.betavariate(*ps[m]))
        return m, f"trying {LABELS[m].lower()}: a new way to learn this"
    return best, ''

A weak concept gets the best method. With `explore='never'` the best method always comes. With `sometimes`, a method that got better is tried at times, and the best method still comes most of the time.

In [ ]:
cs = dict(methods={'*': {'code': [7., 2.], 'maths': [2., 4.]}}, recent={'maths': [0., 0., 1., 1., 1., 1.]})
m, note = choose(cs, 'ml', weak=True)
test_eq(m, 'code'); assert note.startswith('drill with code: 7 of 9')
test_eq({choose(cs, 'ml', rng=random.Random(i), explore='never')[0] for i in range(30)}, {'code'})
picks = [choose(cs, 'ml', rng=random.Random(i)) for i in range(60)]
assert any(n.startswith('trying maths: you got better') for _, n in picks)
test_eq(sum(m == 'code' for m, _ in picks) > 30, True)   # with `sometimes`, the best method still comes most of the time

## In plain words, and from the attempts

`methods_text` says what works, for a prompt and for the page. `method_stats` builds the counts again from the attempts, in order.

In [ ]:
#| export
def methods_text(st, subject:str='', n:int=3) -> str:
    "The methods with counts, best first, as one line; empty when nothing is counted."
    ms = {m: ab for m, ab in ((st.get('methods') or {}).get(subject or '*') or {}).items() if sum(ab) >= 1}
    if not ms: return ''
    top = sorted(ms, key=lambda m: -mean(posterior(st, m, subject)))[:n]
    bits = [f"{LABELS[m].lower()} {round(ms[m][0])} of {round(sum(ms[m]))}" + (' (better lately)' if improving(st, m) else '') for m in top]
    return 'Ways that work for you: ' + '; '.join(bits) + '.'

@patch
def method_stats(self:Learning, n:int=2000) -> dict:
    "Count every final attempt with a method again, oldest first, and save the counts. Return the stats."
    st = {**EMPTY, 'calib': self.stats().get('calib') or {}}
    for a in reversed(self.attempts(limit=n)):
        if a.get('final') and a.get('method') and not a.get('dont_know'):
            c = self.concept(a.concept)
            st = credit(st, a.method, (c or {}).get('subject', ''), a.get('outcome', ''))
    with _LOCK: self.set_stats(**st)
    return st

The counts that `credit_method` keeps match a rebuild from the attempts.

In [ ]:
ml_ = temp_learning(); ml_.put('Ranking', 'Order items by a score.', 'ml')
for m, k in (('code', 'solid'), ('code', 'fragile'), ('maths', 'gap')):
    ml_.record('ranking', f'{m} {k}?', final=True, method=m, outcome=k); ml_.credit_method(m, 'ml', k); time.sleep(.01)
inc = ml_.stats()
test_eq(ml_.method_stats()['methods'], inc.methods)
assert methods_text(inc, 'ml').startswith('Ways that work for you: code 2 of 2')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()